# **Used Collab to train the model because of the local GPU bottleneck**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# After mounting, you can copy your zip file to /content/
# This assumes frames.zip is in the root of your Google Drive's 'My Drive' folder.
# Adjust the source path if it's in a different location.
!cp "/content/drive/MyDrive/frames.zip" "/content/frames.zip"

Mounted at /content/drive


In [9]:
import shutil
import os

# Path to the frames directory
frames_path = '/content/frames/'

if os.path.exists(frames_path):
    shutil.rmtree(frames_path)
    print(f"Successfully deleted: {frames_path}")
else:
    print(f"Directory not found: {frames_path}")

Successfully deleted: /content/frames/


In [10]:
import zipfile
with zipfile.ZipFile('/content/frames.zip', 'r') as z:
    z.extractall('/content/')
print("Done unzipping")

!pip install torch torchvision -q
print("Libraries ready")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import resnet50, ResNet50_Weights
import os
import json

# ── CONFIGURATION ──────────────────────────────────────────
FRAMES_DIR = "/content/frames/"
NUM_CLASSES = 8
BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 0.0001    # Reverted to original stable learning rate
IMAGE_SIZE = 224
SAVE_PATH = "factory_model.pth"

# ── DATA TRANSFORMS ────────────────────────────────────────
# Reverted to simpler augmentation which performed better (62%)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ── LOAD DATASET ───────────────────────────────────────────
train_dataset = datasets.ImageFolder(root=os.path.join(FRAMES_DIR, "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(root=os.path.join(FRAMES_DIR, "test"), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ── BUILD MODEL ────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = resnet50(weights=ResNet50_Weights.DEFAULT)

# Reverted Fix 1: Freeze backbone, only train Layer4 and FC (worked better)
for name, param in model.named_parameters():
    if "layer4" not in name and "fc" not in name:
        param.requires_grad = False

# Reverted Fix 3: Standard FC layer (removed manual dropout to match 62% version)
model.fc = nn.Linear(2048, NUM_CLASSES)
model = model.to(device)

# ── LOSS AND OPTIMIZER ─────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# ── TRAINING LOOP ──────────────────────────────────────────
best_val_acc = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 6

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss, train_correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()

    train_acc = 100 * train_correct / len(train_dataset)
    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100 * val_correct / len(val_dataset)
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)

    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.1f}% | Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.1f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'num_classes': NUM_CLASSES,
        }, SAVE_PATH)
        print(f"  ✓ New best model saved! Val acc: {val_acc:.1f}%")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nTraining complete! Best validation accuracy: {best_val_acc:.1f}%")

idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}
with open("class_mapping.json", "w") as f:
    json.dump(idx_to_class, f, indent=2)

from google.colab import files
files.download('factory_model.pth')
files.download('class_mapping.json')

Done unzipping
Libraries ready
Epoch [01/20] Train Loss: 0.9253 Acc: 66.3% | Val Loss: 0.9951 Acc: 56.8%
  ✓ New best model saved! Val acc: 56.8%
Epoch [02/20] Train Loss: 0.6358 Acc: 75.4% | Val Loss: 0.9420 Acc: 60.4%
  ✓ New best model saved! Val acc: 60.4%
Epoch [03/20] Train Loss: 0.5346 Acc: 78.3% | Val Loss: 1.0257 Acc: 55.9%
Epoch [04/20] Train Loss: 0.4680 Acc: 80.7% | Val Loss: 1.1080 Acc: 56.8%
Epoch [05/20] Train Loss: 0.4217 Acc: 82.1% | Val Loss: 1.3954 Acc: 56.4%
Epoch [06/20] Train Loss: 0.3804 Acc: 83.5% | Val Loss: 1.2985 Acc: 57.3%
Epoch [07/20] Train Loss: 0.3034 Acc: 87.2% | Val Loss: 1.5327 Acc: 58.3%
Epoch [08/20] Train Loss: 0.2721 Acc: 88.3% | Val Loss: 1.7787 Acc: 56.9%

Early stopping at epoch 8

Training complete! Best validation accuracy: 60.4%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>